# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    # Add each new todo description from the provided list to the global todos list.
    todos.extend(descriptions)
    # For each new todo added, append a False value to the completed list, 
    # indicating that none of the new todos are completed yet.
    completed.extend([False] * len(descriptions))
    # Generate and return a formatted report of all current todos, 
    # marking those already completed as appropriate.
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    # Check if the provided index is within the valid range of the todos list (1-based indexing)
    if 1 <= index <= len(todos):
        # Mark the todo at the given index as completed by setting the corresponding flag to True
        completed[index - 1] = True
    else:
        # If index is not valid, return an error message indicating no todo exists at this position
        return "No todo at this index."
    # Print the completion notes using rich Console syntax for display in colored/marked-up format
    Console().print(completion_notes)
    # Generate and return the current todo report, reflecting the updated completed status
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
# JSON schema/function descriptor for the create_todos tool used for LLM function calling.
# This dictionary tells an LLM agent (e.g., OpenAI functions tools) how to invoke our `create_todos` function:
# - "name": the name the LLM should use to call the function
# - "description": a human-readable explanation of the purpose of the function
# - "parameters": a JSON schema describing the function's expected parameter structure:
#     - type: only accepts an object
#     - properties: must have a field "descriptions" which is an array of strings
#     - required: "descriptions" is mandatory
#     - additionalProperties: False prevents extra non-specified properties
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
# JSON schema/function descriptor for the mark_complete tool used for LLM function calling.
# This dictionary tells the LLM agent (e.g., OpenAI functions tools) how to call our mark_complete function:
# - "name": the string name the LLM should use to reference the function.
# - "description": a natural language summary of what the function does, suitable for an AI agent.
# - "parameters": a JSON schema that describes the structure and type of the function's expected arguments:
#     - type: should be "object" (indicating an object/dict supplied to the function)
#     - properties: a dict describing each parameter:
#         - "index": (integer) The 1-based position of the todo item to mark as complete
#             - description: details the expected meaning of the index
#             - title: a short human-readable name for the parameter
#             - type: type of the parameter (integer)
#         - "completion_notes": (string) Rich markup notes about how the todo was completed
#             - description: what should be provided for this string field
#             - title: human-readable parameter name
#             - type: type of the parameter (string)
#     - required: which parameters must be present ("index" and "completion_notes" are both mandatory)
#     - additionalProperties: False, enforcing that no extra arguments are allowed
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'type': 'object',
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
            },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        'required': ['index', 'completion_notes'],
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    # Initialize an empty list to store the results for each tool call
    results = []
    # Iterate over the list of tool calls received from the LLM
    for tool_call in tool_calls:
        # Extract the function (tool) name that the LLM wants to invoke
        tool_name = tool_call.function.name
        # Parse the arguments for the tool from a JSON string to a native Python dict
        arguments = json.loads(tool_call.function.arguments)
        # Dynamically fetch the tool (function) from the global namespace using its name
        tool = globals().get(tool_name)
        # If the tool was found, call it with the parsed arguments; otherwise, use an empty result
        result = tool(**arguments) if tool else {}
        # Append the result to the results list, formatted for LLM consumption:
        # - role: always "tool" (indicates this is a tool's reply)
        # - content: the tool result as a JSON string (for LLM context)
        # - tool_call_id: links this reply to the originating tool call for associating responses
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    # Return the list of results (tool responses) to be added to the LLM conversation
    return results

In [15]:
def loop(messages):
    # Flag to keep track of when the loop should terminate
    done = False
    while not done:
        # Send the current conversation messages, tool schema, etc. to the LLM
        response = openai.chat.completions.create(
            model="gpt-5.2",
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )
        # Get the reason why the LLM stopped its current turn
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "tool_calls":
            # The LLM responded with a request to use tool(s)
            message = response.choices[0].message
            tool_calls = message.tool_calls  # Extract the list of tool function calls
            # Execute all tools the LLM asked for and collect their results
            results = handle_tool_calls(tool_calls)
            # Add the LLM's message (the tool call request itself) to the ongoing conversation
            messages.append(message)
            # Add the tool call results as "tool" messages to the conversation – so the LLM sees outputs next round
            messages.extend(results)
        else:
            # LLM has finished its reasoning/output, so exit the loop
            done = True
    # After breaking out of loop, display the final answer from the LLM (without tool calls)
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Parse the problem, define variables, and note any missing quantities (distance between Boston and New 
York).
Todo #2: Provide a reasonable estimate for the Boston–New York distance and state it clearly.
Todo #3: Compute each train’s position as a function of time and solve for the meeting time.
Todo #4: Convert the solution into a clock time (pm) and present the final answer clearly.

Let D be the distance between Boston and New York (not given). Measure time t in hours after 2:00 pm.
- Boston train speed: 60 mph, starts at t=0.
- New York train speed: 80 mph, starts at 3:00 pm = t=1.
They meet when distances traveled add to D.

Todo #1: Parse the problem, define variables, and note any missing quantities (distance between Boston and New 
York).
Todo #2: Provide a reasonable estimate for the Boston–New York distance and state it clearly.
Todo #3: Compute each train’s position as a function of time and solve for the meeting time.
Todo #4: Convert the solution into a clock time (pm) and present the final answer clearly.

Use a reasonable rail/road distance estimate between Boston and New York: about 215 miles (commonly ~215–220 mi by 
road; rail is similar order). Take D = 215 mi.

Todo #1: Parse the problem, define variables, and note any missing quantities (distance between Boston and New 
York).
Todo #2: Provide a reasonable estimate for the Boston–New York distance and state it clearly.
Todo #3: Compute each train’s position as a function of time and solve for the meeting time.
Todo #4: Convert the solution into a clock time (pm) and present the final answer clearly.

For t ≥ 1:
- Boston train distance: 60t.
- NY train distance: 80(t−1).
Meet when 60t + 80(t−1) = D.
So 140t − 80 = D ⇒ t = (D+80)/140.
With D=215: t = (215+80)/140 = 295/140 ≈ 2.1071 hours after 2:00 pm.

Todo #1: Parse the problem, define variables, and note any missing quantities (distance between Boston and New 
York).
Todo #2: Provide a reasonable estimate for the Boston–New York distance and state it clearly.
Todo #3: Compute each train’s position as a function of time and solve for the meeting time.
Todo #4: Convert the solution into a clock time (pm) and present the final answer clearly.

t ≈ 2.1071 hr after 2:00 pm = 2 hr + 0.1071 hr.
0.1071 hr × 60 ≈ 6.43 min.
Meeting time ≈ 4:06 pm (about 4:06:26 pm).

Todo #1: Parse the problem, define variables, and note any missing quantities (distance between Boston and New 
York).
Todo #2: Provide a reasonable estimate for the Boston–New York distance and state it clearly.
Todo #3: Compute each train’s position as a function of time and solve for the meeting time.
Todo #4: Convert the solution into a clock time (pm) and present the final answer clearly.

Missing information: the distance between Boston and New York isn’t given, so an exact time can’t be determined 
without it.

Using a reasonable estimate D ≈ 215 miles:

Let t = hours after 2:00 pm when they meet (so the NY train has traveled for t−1 hours).

- Boston train distance: 60t
- New York train distance: 80(t−1)

Meet when:
60t + 80(t−1) = 215  
140t − 80 = 215 → t = (215+80)/140 = 295/140 ≈ 2.107 hours after 2:00 pm.

2.107 hours = 2 hours + 0.107×60 ≈ 2 hours + 6.4 minutes

They meet at about 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>